**1. Imports**

In [1]:
import pandas as pd
import numpy as np

**2. Load Data**

In [2]:
df = pd.read_csv("../data/raw/Crop Prediction dataset.csv")

**3. Standardize Column Names**

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns

Index(['state_name', 'district_name', 'crop_year', 'season', 'crop',
       'temperature', 'humidity', 'soil_moisture', 'area', 'production'],
      dtype='object')

**4. Identify Target Column**

In [4]:
possible_targets = ['label', 'crop', 'crop_type']

target_col = None
for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

print("Target Column:", target_col)

Target Column: crop


**5. Check Missing Values**

In [5]:
df.isnull().sum()

state_name         0
district_name      0
crop_year          0
season             0
crop               0
temperature        0
humidity           0
soil_moisture      0
area               0
production       215
dtype: int64

**Handle Missing**

In [6]:
df = df.fillna(df.median(numeric_only=True))

In [7]:
df.isnull().sum()

state_name       0
district_name    0
crop_year        0
season           0
crop             0
temperature      0
humidity         0
soil_moisture    0
area             0
production       0
dtype: int64

**6. Remove Duplicates**

In [8]:
df = df.drop_duplicates()
print("New Shape:", df.shape)

New Shape: (49999, 10)


**7. Fix Data Types**

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49999 entries, 0 to 49998
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   state_name     49999 non-null  object 
 1   district_name  49999 non-null  object 
 2   crop_year      49999 non-null  int64  
 3   season         49999 non-null  object 
 4   crop           49999 non-null  object 
 5   temperature    49999 non-null  int64  
 6   humidity       49999 non-null  int64  
 7   soil_moisture  49999 non-null  int64  
 8   area           49999 non-null  float64
 9   production     49999 non-null  float64
dtypes: float64(2), int64(4), object(4)
memory usage: 3.8+ MB


**8. Outlier Handling**

In [10]:
def remove_outliers(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    
    return df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

numeric_cols = df.select_dtypes(include=np.number).columns

df_clean = remove_outliers(df[numeric_cols])
df = df.loc[df_clean.index]

print("Shape after outlier removal:", df.shape)

Shape after outlier removal: (34920, 10)


**9. Feature / Target Split**

In [11]:
X = df.drop(target_col, axis=1)
y = df[target_col]

**10. Encode Target**

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [13]:
df.head()

,state_name,district_name,crop_year,season,crop,temperature,humidity,soil_moisture,area,production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,36,35,45,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,37,40,46,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,36,41,50,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,37,42,55,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,36,40,54,720.0,165.0


**11. Save Clean Dataset**

In [14]:
df_cleaned = X.copy()
df_cleaned[target_col] = y_encoded

df_cleaned.to_csv("../data/processed/cleaned_crop_data.csv", index=False)

**12. Save Encoder**

In [15]:
import pickle

pickle.dump(le, open("../models/label_encoder.pkl", "wb"))

**13. Final Check**

In [16]:
print(X.shape)
print(y_encoded.shape)
print("Classes:", list(le.classes_))

(34920, 9)
(34920,)
Classes: ['Arecanut', 'Arhar/Tur', 'Bajra', 'Banana', 'Barley', 'Beans & Mutter(Vegetable)', 'Bhindi', 'Black pepper', 'Blackgram', 'Bottle Gourd', 'Brinjal', 'Cabbage', 'Cashewnut', 'Castor seed', 'Citrus Fruit', 'Coconut ', 'Coriander', 'Cotton(lint)', 'Cowpea(Lobia)', 'Cucumber', 'Dry chillies', 'Dry ginger', 'Garlic', 'Ginger', 'Gram', 'Grapes', 'Groundnut', 'Guar seed', 'Horse-gram', 'Jowar', 'Jute', 'Khesari', 'Korra', 'Lemon', 'Linseed', 'Maize', 'Mango', 'Masoor', 'Mesta', 'Moong(Green Gram)', 'Niger seed', 'Oilseeds total', 'Onion', 'Orange', 'Other  Rabi pulses', 'Other Fresh Fruits', 'Other Kharif pulses', 'Other Vegetables', 'Paddy', 'Papaya', 'Peas  (vegetable)', 'Peas & beans (Pulses)', 'Pineapple', 'Pome Fruit', 'Pome Granet', 'Potato', 'Pulses total', 'Ragi', 'Rapeseed &Mustard', 'Rice', 'Safflower', 'Samai', 'Sannhamp', 'Sapota', 'Sesamum', 'Small millets', 'Soyabean', 'Sugarcane', 'Sunflower', 'Sweet potato', 'Tapioca', 'Tobacco', 'Tomato', 'Turmer